In [1]:
import numpy as np
import rl_train.train.train_configs.config as myoassist_config
import rl_train.utils.train_log_handler as train_log_handler
from rl_train.utils.data_types import DictionableDataclass
import json
import os
from datetime import datetime
from rl_train.envs.environment_handler import EnvironmentHandler
import subprocess

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry."

MyoSuite:> Registering Myo Envs


In [ ]:
def get_git_info():
    try:
        commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode('ascii').strip()
        branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode('ascii').strip()
        return {"commit": commit, "branch": branch}
    except:
        return {"commit": "unknown", "branch": "unknown"}

VERSION = {"version": "0.3.0", **get_git_info()}

def ppo_evaluate_with_rendering(config):
    np.random.seed(1234)
    env = EnvironmentHandler.create_environment(config, is_rendering_on=True, is_evaluate_mode=True)
    model = EnvironmentHandler.get_stable_baselines3_model(config, env)
    EnvironmentHandler.updateconfig_from_model_policy(config, model)

    obs, info = env.reset()
    for _ in range(config.evaluate_param_list[0]["num_timesteps"]):
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, done, truncated, info = env.step(action)
        if truncated:
            obs, info = env.reset()
    env.close()

def ppo_train_with_parameters(config, train_time_step, is_rendering_on, train_log_handler, log_dir):
    np.random.seed(1234)

    session_config_dict = DictionableDataclass.to_dict(config)
    session_config_dict["env_params"].pop("reference_data", None)
    session_config_dict["code_version"] = VERSION

    session_config_path = os.path.join(log_dir, 'session_config.json')
    with open(session_config_path, 'w', encoding='utf-8') as f:
        json.dump(session_config_dict, f, ensure_ascii=False, indent=4)

    env = EnvironmentHandler.create_environment(config, is_rendering_on)
    model = EnvironmentHandler.get_stable_baselines3_model(config, env)
    EnvironmentHandler.updateconfig_from_model_policy(config, model)

    custom_callback = EnvironmentHandler.get_callback(config, train_log_handler)
    model.learn(
        reset_num_timesteps=False,
        total_timesteps=train_time_step,
        log_interval=1,
        callback=custom_callback,
        progress_bar=True
    )
    env.close()
    print("learning done!")

CONFIG_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\rl_train\train\train_configs\imitation_tutorial_22_separated_net_partial_obs.json"
RENDERING_ON = False
REALTIME_EVAL = False
MYO_ROOT = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist"

default_config = EnvironmentHandler.get_session_config_from_path(CONFIG_PATH, myoassist_config.TrainSessionConfigBase)
default_config.env_params.reference_data_path = os.path.join(
    MYO_ROOT, "rl_train", "reference_data", "short_reference_gait.npz"
)

config_type = EnvironmentHandler.get_config_type_from_session_id(default_config.env_params.env_id)
config = EnvironmentHandler.get_session_config_from_path(CONFIG_PATH, config_type)
config.env_params.reference_data_path = default_config.env_params.reference_data_path

BASE_LOG_DIR = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\rl_train\results"
log_dir = os.path.join(BASE_LOG_DIR, f"train_session_{datetime.now().strftime('%Y%m%d-%H%M%S')}")
os.makedirs(log_dir, exist_ok=True)
train_log_handler = train_log_handler.TrainLogHandler(log_dir)

config.env_params.model_path = os.path.join(MYO_ROOT, "models", "22muscle_2D", "myoLeg22_2D_TUTORIAL.xml")

In [ ]:
env = EnvironmentHandler.create_environment(config, is_rendering_on=False)

from stable_baselines3 import PPO
last_model_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\rl_train\results\train_session_20250901-203400\trained_models\model_3874816.zip"
if os.path.exists(last_model_path):
    model = PPO.load(last_model_path, env=env)
else:
    model = EnvironmentHandler.get_stable_baselines3_model(config, env)

In [ ]:
# 6. Train
ppo_train_with_parameters(
    config,
    train_time_step=config.total_timesteps,
    is_rendering_on=RENDERING_ON,
    train_log_handler=train_log_handler,
    log_dir=log_dir  # pass absolute log_dir for callback
)

In [ ]:
trained_model_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\rl_train\results\train_session_20250901-203400\trained_models\model_3874816.zip"

CONFIG_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\rl_train\train\train_configs\imitation_tutorial_22_separated_net_partial_obs.json"

default_config = EnvironmentHandler.get_session_config_from_path(
    CONFIG_PATH, 
    myoassist_config.TrainSessionConfigBase
)

# Override reference data path if needed
MYO_ROOT = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist"
default_config.env_params.reference_data_path = os.path.join(
    MYO_ROOT, "rl_train", "reference_data", "short_reference_gait.npz"
)

config_type = EnvironmentHandler.get_config_type_from_session_id(default_config.env_params.env_id)
config = EnvironmentHandler.get_session_config_from_path(CONFIG_PATH, config_type)
config.env_params.reference_data_path = default_config.env_params.reference_data_path


In [ ]:
import os
import sys
import numpy as np
import mujoco
import matplotlib.pyplot as plt
from datetime import datetime
import subprocess
import platform
import glob
from pathlib import Path

from ctrl_optim.ctrl.reflex.reflex_interface import myoLeg_reflex
from ctrl_optim.optim.optim_utils.config_parser import load_params_and_create_testenv, print_config_summary


def print_progress_bar(iteration, total, prefix='', suffix='', decimals=1, length=50, fill='█', print_end="\r"):
    percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = fill * filled_length + '-' * (length - filled_length)
    print(f'\r{prefix} |{bar}| {percent}% {suffix}', end=print_end)
    if iteration == total:
        print()


def open_video_in_new_window(video_path):
    try:
        if platform.system() == "Windows":
            os.startfile(video_path)
        elif platform.system() == "Darwin":
            subprocess.run(["open", video_path])
        else:
            subprocess.run(["xdg-open", video_path])
    except Exception as e:
        print(f"Could not open video: {e}")


def find_config_file(results_dir):
    config_files = []
    for ext in ["*.bat", "*.sh"]:
        config_files.extend(glob.glob(os.path.join(results_dir, ext)))
    if not config_files:
        raise FileNotFoundError(f"No configuration file (.bat or .sh) found in directory: {results_dir}")
    return config_files[0]


def main():
    LOAD_FROM_FILE = True
    notebook_dir = os.getcwd()
    PARAMS_FILE_PATH = os.path.join(notebook_dir,"myoassist", "ctrl_optim", "results", "optim_results","tutorial_example", "myorfl_Kine_2D_1_25_2025Aug11_1439_None_BestLast.txt")

    if LOAD_FROM_FILE:
        if not os.path.exists(PARAMS_FILE_PATH):
            raise FileNotFoundError(f"Parameter file not found: {PARAMS_FILE_PATH}")

    SIMULATION_TIME = 6
    SLOPE_DEG = 0
    MODEL = "hmedi"
    EXO_BOOL = True
    USE_4PARAM_SPLINE = True
    N_POINTS = 4
    MAX_TORQUE = 100

    output_folder = "results/evaluation_outputs"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_output_folder = os.path.join(output_folder, f"run_ctrl_{timestamp}")
    os.makedirs(run_output_folder)

    if LOAD_FROM_FILE:
        if not os.path.exists(PARAMS_FILE_PATH):
            raise FileNotFoundError(f"Parameter file not found: {PARAMS_FILE_PATH}")
        
        results_dir = os.path.dirname(PARAMS_FILE_PATH)
        filename = os.path.basename(PARAMS_FILE_PATH)
        config_file_path = find_config_file(results_dir)
        
        env, config, _ = load_params_and_create_testenv(
            results_dir=results_dir,
            filename=filename,
            bat_file_path=config_file_path,
            sim_time=SIMULATION_TIME
        )
    else:
        config = {
            'mode': '2D', 'init_pose': 'walk_left', 'delayed': False,
            'slope_deg': SLOPE_DEG,
            'model': MODEL,
            'exo_bool': EXO_BOOL,
            'use_4param_spline': USE_4PARAM_SPLINE,
            'n_points': N_POINTS,
            'max_torque': MAX_TORQUE
        }
        
        if config['exo_bool']:
            spline_params = 4 if config['use_4param_spline'] else (config['n_points'] * 2)
        else:
            spline_params = 0
        control_params = np.ones(77 + spline_params)
        
        env = myoLeg_reflex(
            sim_time=SIMULATION_TIME,
            control_params=control_params,
            **config
        )
    env.reset()

    env.reset()
    timesteps = int(SIMULATION_TIME / env.dt)
    frames = []

    video_width, video_height = 1920, 1080
    env.env.sim.renderer.render_offscreen(camera_id=4, width=video_width, height=video_height)
    env.env.sim.renderer._scene_option.flags[0] = 0
    env.env.sim.renderer._scene_option.flags[4] = 0

    free_cam = mujoco.MjvCamera()
    camera_speed = 1.25
    slope_angle_rad = np.radians(env.slope_deg)
    start_position = env.env.unwrapped.sim.data.body("pelvis").xpos.copy()
    
    camera_pos = start_position.copy()
    camera_pos[2] = 0.8

    for i in range(timesteps):
        if i % max(1, timesteps // 10) == 0 or i % 50 == 0:
            print_progress_bar(i, timesteps, prefix='Progress:', suffix=f'({i}/{timesteps})', length=30)
        
        if not env.delayed:
            distance_traveled = camera_speed * env.dt * i
            camera_pos[0] = start_position[0] + distance_traveled
            
            slope_correction = 0.2
            height_increase = (camera_pos[0] - start_position[0]) * np.tan(slope_angle_rad) * slope_correction
            camera_pos[2] = 0.8 + height_increase
            
            pelvis_pos = env.env.unwrapped.sim.data.body("pelvis").xpos.copy()
            lookat_pos = camera_pos.copy()
            lookat_pos[1] = pelvis_pos[1]
            
            free_cam.distance = 2.5
            free_cam.azimuth = 90
            free_cam.elevation = 0
            free_cam.lookat = lookat_pos
            
            frame = env.env.unwrapped.sim.renderer.render_offscreen(
                    camera_id=free_cam, width=video_width, height=video_height)
        else:
            if i % 10 == 0:
                env.env.sim.data.camera(4).xpos[2] = 2.181
            frame = env.env.sim.renderer.render_offscreen(camera_id=4, width=video_width, height=video_height)
        
        if frame is not None and len(frame.shape) == 3:
            actual_height, actual_width = frame.shape[:2]
            if i == 0:
                if actual_width != video_width or actual_height != video_height:
                    video_width, video_height = actual_width, actual_height
        frames.append(frame)
        _, _, is_done = env.run_reflex_step_Cost()
        
        if is_done:
            break
    
    print_progress_bar(timesteps, timesteps, prefix='Progress:', suffix=f'({timesteps}/{timesteps})', length=30)

    video_filename = "simulation_regular.mp4"
    video_path = os.path.join(run_output_folder, video_filename)
    
    try:
        import imageio
        with imageio.get_writer(video_path, fps=100, codec='libx264', quality=9, macro_block_size=None) as writer:
            for frame in frames:
                if frame is not None:
                    if frame.max() <= 1.0:
                        frame = (frame * 255).astype(np.uint8)
                    else:
                        frame = frame.astype(np.uint8)
                    writer.append_data(frame)
    except Exception as e:
        print(f"Error: Video failed: {e}")

    open_video_in_new_window(video_path)


if __name__ == "__main__":
    main()

In [ ]:
LOAD_FROM_FILE = False

SIMULATION_TIME = 5
SLOPE_DEG = 0
MODEL = "baseline"
EXO_BOOL = False
USE_4PARAM_SPLINE = False
N_POINTS = 4
MAX_TORQUE = 100

if LOAD_FROM_FILE:
    env, config, _ = load_params_and_create_testenv(
        results_dir=results_dir,
        filename=filename,
        bat_file_path=bat_file_path,
        sim_time=SIMULATION_TIME
    )
else:
    config = {
        'mode': '2D', 'init_pose': 'walk_left', 'delayed': False,
        'slope_deg': SLOPE_DEG, 'model': MODEL, 'exo_bool': EXO_BOOL,
        'use_4param_spline': USE_4PARAM_SPLINE, 'n_points': N_POINTS,
        'max_torque': MAX_TORQUE
    }
    
    if config['exo_bool']:
        spline_params = 4 if config['use_4param_spline'] else (config['n_points'] * 2)
    else:
        spline_params = 0
    control_params = np.ones(77 + spline_params)
    
    env = myoLeg_reflex(sim_time=SIMULATION_TIME, control_params=control_params, **config)

In [ ]:
control_params = np.random.randn(77)
env = myoLeg_reflex(
    sim_time=10,
    control_params=control_params,
    mode='2D',
    init_pose='walk_left',
    delayed=False,
    slope_deg=0,
    model='baseline',
    exo_bool=False
)
env.reset()
env.env.mj_render()

In [ ]:
timesteps = int(5 / env.dt)
walking_duration = 0
env.env.mj_render()
for i in range(timesteps):
    _, _, is_done = env.run_reflex_step_Cost()
    walking_duration = (i + 1) * env.dt
    if is_done:
        break

print(f"Walking duration: {walking_duration:.3f} seconds")

In [ ]:
from ctrl_optim.optim.optim_utils.fati_config_parser import load_params_and_create_testenv, print_config_summary

LOAD_FROM_FILE = True
PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\baseline_0902_2219\myorfl_Kine_2D_1_25_2025Sep02_2219_None_BestLast.txt"

SIMULATION_TIME = 5
SLOPE_DEG = 0
MODEL = "baseline"
EXO_BOOL = False
USE_4PARAM_SPLINE = False
N_POINTS = 4
MAX_TORQUE = 100
results_dir = os.path.dirname(PARAMS_FILE_PATH)
filename = os.path.basename(PARAMS_FILE_PATH)
bat_file_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\optim\training_configs\baseline_cie.bat"
if LOAD_FROM_FILE:
    env, config, _ = load_params_and_create_testenv(
        results_dir=results_dir,
        filename=filename,
        bat_file_path=bat_file_path,
        sim_time=SIMULATION_TIME
    )

In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs = env.reset()
        total_reward = 0
        episode_length = 100

        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            total_reward += reward
            if step % 20 == 0:
                env.render()
            if done:
                break
        env.close()
        return True
        
    except Exception as e:
        print(f"Error during walk test: {e}")
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs = env.reset()
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            if done:
                break
        env.close()
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

def register_terrain_variants():
    gym.register(
        id="myoLegCustomRoughTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "rough",
            "variant": None,
        },
    )
    gym.register(
        id="myoLegCustomHillyTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "hilly",
            "variant": "fixed",
        },
    )
    gym.register(
        id="myoLegCustomStairTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "stairs",
            "variant": "fixed",
        },
    )

if __name__ == "__main__":
    basic_success = quick_walk_test()
    if basic_success:
        test_walk_environment()

Custom environment 'myoLegCustomWalk-v0' registered successfully!
Model: 80-muscle leg model (HMEDI)
Task: Bipedal walking

Testing MyoLeg 80-Muscle Custom Walk Environment...
PATH DEBUGGING
Checking path: C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml
Path exists: True
Is file: True
Parent directory exists: True
Files in parent directory:
  - assets
  - myolegs_HMEDI.xml

Alternative path format: C:/Users/rohan/Downloads/Exoskeleton_PD/exo_leg/myoassist/models/80muscle/myoLeg80_HMEDI/myolegs_HMEDI.xml
Alternative path exists: True

QUICK WALK TEST (NO RENDERING)
Environment created successfully
Action space: (82,)
Observation space: (409,)
Number of muscles: 80
Step 1: reward=6.5442, done=False
Step 2: reward=4.4275, done=False
Step 3: reward=2.8883, done=False
Step 4: reward=1.6525, done=False
Step 5: reward=0.1376, done=False
Step 6: reward=-1.3068, done=False
Step 7: reward=-2.2259, done=False
Step 8: reward=-2.2962, done=

c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.sim to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.sim` for environment variables or `env.get_wrapper_attr('sim')` that will search the reminding wrappers.
  logger.warn(


In [24]:
import gymnasium as gym
env = gym.make("myoLegCustomReach-v0")
import time
env.mj_render()
time.sleep(2)
model = PPO.load('myoReach-22', env=env, verbose = 1)
obs, _ = env.reset()
tr=0
done = False
while done == False:
    action, _ = model.predict(obs, deterministic=True)  # 22-dimensional action
    obs, reward, done, _, info = env.step(action)
    tr += reward
    env.mj_render()  # 134-dimensional observation
print("tr", tr)

c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.mj_render to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.mj_render` for environment variables or `env.get_wrapper_attr('mj_render')` that will search the reminding wrappers.
  logger.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
tr -48.059146734422264


In [ ]:
from stable_baselines3 import PPO
from myosuite.utils import gym

model = PPO.load("LegWalk_Policy", env, verbose=1)

model.learn(total_timesteps=1000000)

model.save('LegWalk_policy')

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./ppo_leg_exo_tensorboard/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10.7     |
|    ep_rew_mean     | -51.3    |
| time/              |          |
|    fps             | 114      |
|    iterations      | 1        |
|    time_elapsed    | 17       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 10.8        |
|    ep_rew_mean          | -51.6       |
| time/                   |             |
|    fps                  | 110         |
|    iterations           | 2           |
|    time_elapsed         | 36          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010506018 |
|    clip_fraction        | 0.125       |
|    clip_range    

In [ ]:
ppo_reflex_exo.save('myoReach-22')

In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        
        action_space = env.action_space
        obs_space = env.observation_space
        num_muscles = env.sim.model.na
        
        obs = env.reset()
        
        total_reward = 0
        episode_length = 100
        
        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            total_reward += reward
            
            if step % 20 == 0:
                env.render()
            
            time.sleep(0.01)
            
            if done:
                break
        
        env.close()
        return True
        
    except Exception as e:
        print(f"Error: {e}")
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs = env.reset()
        
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            if done:
                break
        
        env.close()
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

def register_terrain_variants():
    gym.register(
        id="myoLegCustomRoughTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "rough",
            "variant": None,
        },
    )
    gym.register(
        id="myoLegCustomHillyTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "hilly",
            "variant": "fixed",
        },
    )
    gym.register(
        id="myoLegCustomStairTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "stairs",
            "variant": "fixed",
        },
    )

if __name__ == "__main__":
    basic_success = quick_walk_test()
    if basic_success:
        test_walk_environment()

In [3]:
import gymnasium as gym
env = gym.make("myoLegCustomWalk-v0")
import time
env.mj_render()
time.sleep(2)
#model = PPO.load('myoReach-22', env=env, verbose = 1)
obs, _ = env.reset()
for i in range(200):
    action = env.action_space.sample()  # 22-dimensional action
    obs, reward, done, _, info = env.step(action)
    env.mj_render()  # 134-dimensional observation

c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.mj_render to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.mj_render` for environment variables or `env.get_wrapper_attr('mj_render')` that will search the reminding wrappers.
  logger.warn(


In [ ]:
from myosuite.utils import gym
import skvideo.io
import numpy as np
import os
import git
import skvideo
skvideo.setFFmpegPath(r"C:\Users\rohan\Downloads\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\bin")
import myosuite
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
import time
from IPython.display import HTML
from base64 import b64encode
import PIL.Image, PIL.ImageDraw

def show_video(video_path, video_width=400):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

def add_text_to_frame(frame, text, pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return frame

os.environ["GIT_PYTHON_REFRESH"] = "quiet"
from stable_baselines3 import PPO
import gymnasium as gym

c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\skvideo\__init__.py:306: UserWarning: ffmpeg/ffprobe not found in path: C:\Users\rohan\Downloads\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\bin
  warnings.warn("ffmpeg/ffprobe not found in path: " + str(path), UserWarning)


In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs = env.reset()
        total_reward = 0
        episode_length = 100

        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            total_reward += reward
            if step % 20 == 0:
                env.render()
            if done:
                break
        env.close()
        return True
        
    except Exception as e:
        print(f"Error: {e}")
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs = env.reset()
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            if done:
                break
        env.close()
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

def register_terrain_variants():
    gym.register(
        id="myoLegCustomRoughTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "rough",
            "variant": None,
        },
    )
    gym.register(
        id="myoLegCustomHillyTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "hilly",
            "variant": "fixed",
        },
    )
    gym.register(
        id="myoLegCustomStairTerrainWalk-v0",
        entry_point="myosuite.envs.myo.myobase.walk_v0:TerrainEnvV0",
        max_episode_steps=1000,
        kwargs={
            "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
            "normalize_act": True,
            "min_height": 0.8,
            "max_rot": 0.8,
            "hip_period": 100,
            "reset_type": "init",
            "target_x_vel": 0.0,
            "target_y_vel": 1.2,
            "target_rot": None,
            "terrain": "stairs",
            "variant": "fixed",
        },
    )

if __name__ == "__main__":
    basic_success = quick_walk_test()
    if basic_success:
        test_walk_environment()

Custom environment 'myoLegCustomWalk-v0' registered successfully!
Model: 80-muscle leg model (HMEDI)
Task: Bipedal walking

Testing MyoLeg 80-Muscle Custom Walk Environment...

QUICK WALK TEST (NO RENDERING)
Environment created successfully
Action space: (82,)
Observation space: (409,)
Number of muscles: 80
✗ Quick walk test failed: too many values to unpack (expected 4)

❌ Basic functionality test failed.
Please check your XML file and ensure it has walking-compatible structure.


c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoLegCustomWalk-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [11]:
print(env.action_space)
print(env.observation_space)

Box(-1.0, 1.0, (80,), float32)
Box(-10.0, 10.0, (403,), float32)


In [ ]:
from stable_baselines3 import PPO
from myosuite.utils import gym

model = PPO.load("LegWalk_Policy", env, verbose=1)

model.learn(total_timesteps=1000000)

model.save('LegWalk_policy')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Starting policy learning
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 126      |
|    ep_rew_mean     | 726      |
| time/              |          |
|    fps             | 347      |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 116         |
|    ep_rew_mean          | 695         |
| time/                   |             |
|    fps                  | 321         |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.066685274 |
|    clip_fraction        | 0.525       |
|    clip_range           | 0.2         |
|    entropy

In [ ]:
import numpy as np
from gymnasium import ObservationWrapper, ActionWrapper, spaces

class ObsAdapter(ObservationWrapper):
    def __init__(self, env, expected_obs_dim=403):
        super().__init__(env)
        self.expected_obs_dim = expected_obs_dim
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(expected_obs_dim,), dtype=np.float32
        )

    def observation(self, obs):
        if obs.shape[0] > self.expected_obs_dim:
            return obs[:self.expected_obs_dim]
        elif obs.shape[0] < self.expected_obs_dim:
            return np.pad(obs, (0, self.expected_obs_dim - obs.shape[0]))
        return obs


class ActionAdapter(ActionWrapper):
    def __init__(self, env, expected_action_dim=80):
        super().__init__(env)
        self.expected_action_dim = expected_action_dim
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(expected_action_dim,), dtype=np.float32
        )

    def action(self, act):
        env_dim = self.env.action_space.shape[0]
        if len(act) > env_dim:
            return act[:env_dim]
        elif len(act) < env_dim:
            return np.pad(act, (0, env_dim - len(act)))
        return act

In [ ]:
from myosuite.utils import gym
from stable_baselines3 import PPO
import time

env = gym.make('myoLegCustomWalk-v0')

env = ObsAdapter(env, expected_obs_dim=403)
env = ActionAdapter(env, expected_action_dim=80)

obs, _ = env.reset()

model = PPO.load("LegWalk_policy", env, verbose=1)

done = False
tr = 0
env.mj_render()
time.sleep(3)

while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _, info = env.step(action)
    env.mj_render()
    tr += reward

print("Final reward", tr)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry."

MyoSuite:> Registering Myo Envs


NameNotFound: Environment `myoLegCustomWalk` doesn't exist. Did you mean: `myoLegWalk`?